# Notebook 02 – EDA: Khám Phá 4 Datasets

**Mục tiêu:** Phân tích thống kê và trực quan hoá toàn bộ 4 datasets của hệ thống.

| Dataset | File | Rows | Cols | Mô tả |
|---------|------|------|------|---------|
| Fundamental | `dataset_fundamental.csv` | 240 | 18 | Báo cáo tài chính 30 tickers × 8 quý |
| Technical | `dataset_technical.csv` | 15,120 | 29 | OHLCV + 20 chỉ báo kỹ thuật |
| Screening | `dataset_screening.csv` | 30 | 23 | Snapshot universe + composite score |
| Orchestrator | `dataset_orchestrator.csv` | 140 | 7 | Queries có nhãn cho RL training |

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '../data/'

# Load tất cả datasets
df_fund = pd.read_csv(DATA_DIR + 'dataset_fundamental.csv')
df_tech = pd.read_csv(DATA_DIR + 'dataset_technical.csv')
df_screen = pd.read_csv(DATA_DIR + 'dataset_screening.csv')
df_orch = pd.read_csv(DATA_DIR + 'dataset_orchestrator.csv')

print('Datasets loaded:')
for name, df in [('fundamental', df_fund), ('technical', df_tech),
                  ('screening', df_screen), ('orchestrator', df_orch)]:
    print(f'  {name:<15}: {df.shape[0]:>6} rows × {df.shape[1]:>2} cols')

---
## PHẦN 1: EDA – Dataset Fundamental

Báo cáo tài chính 30 mã HOSE, 8 quý (2023-Q1 → 2024-Q4)

In [ ]:
print('=== FUNDAMENTAL DATASET ===')
print(f'Shape: {df_fund.shape}')
print(f'Tickers: {df_fund["ticker"].nunique()} mã')
print(f'Quarters: {sorted(df_fund["quarter"].unique())}')
print(f'Sectors: {df_fund["sector"].nunique()} ngành')
print()
df_fund.head()

In [ ]:
# Missing values
print('--- Missing Values ---')
missing = df_fund.isnull().sum()
missing_pct = (missing / len(df_fund) * 100).round(1)
miss_df = pd.DataFrame({'count': missing, 'pct': missing_pct})
miss_df = miss_df[miss_df['count'] > 0]
if len(miss_df):
    print(miss_df)
else:
    print('  Không có missing values.')
print(f'  Completeness tổng: {(1 - df_fund.isnull().sum().sum() / df_fund.size) * 100:.1f}%')

In [ ]:
# Descriptive statistics
print('--- Descriptive Statistics (numeric) ---')
num_cols = ['pe_ratio', 'roe', 'eps_vnd', 'debt_to_equity', 'net_margin', 'gross_margin',
            'revenue_bn_vnd', 'dividend_yield']
desc = df_fund[num_cols].describe().T
desc['skew'] = df_fund[num_cols].skew().round(3)
desc['kurt'] = df_fund[num_cols].kurtosis().round(3)
print(desc[['mean', 'std', 'min', '25%', '50%', '75%', 'max', 'skew', 'kurt']].round(3).to_string())

In [ ]:
# EDA insight 1: Phân phối PE theo sector
print('--- PE Ratio by Sector ---')
pe_by_sector = df_fund.groupby('sector')['pe_ratio'].agg(['mean', 'std', 'min', 'max']).round(2)
pe_by_sector.columns = ['PE_mean', 'PE_std', 'PE_min', 'PE_max']
print(pe_by_sector.sort_values('PE_mean', ascending=False).to_string())

In [ ]:
# EDA insight 2: ROE by sector
print('--- ROE by Sector (mean, %) ---')
roe_by_sector = df_fund.groupby('sector')['roe'].agg(['mean', 'std']).round(4) * 100
roe_by_sector.columns = ['ROE_mean_%', 'ROE_std_%']
print(roe_by_sector.sort_values('ROE_mean_%', ascending=False).to_string())

In [ ]:
# EDA insight 3: Correlation matrix
print('--- Correlation Matrix (key metrics) ---')
corr_cols = ['pe_ratio', 'roe', 'debt_to_equity', 'net_margin', 'gross_margin',
             'eps_vnd', 'dividend_yield']
corr = df_fund[corr_cols].corr().round(3)
print(corr.to_string())
print()
# Highlight strong correlations
print('--- Strong Correlations (|r| > 0.5) ---')
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        r = corr.iloc[i, j]
        if abs(r) > 0.5:
            print(f'  {corr.columns[i]:20s} ↔ {corr.columns[j]:20s}: r = {r:.3f}')

In [ ]:
# EDA insight 4: Outlier detection (IQR method)
print('--- Outliers (IQR method) ---')
for col in ['pe_ratio', 'eps_vnd', 'revenue_bn_vnd', 'debt_to_equity']:
    s = df_fund[col].dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((s < lo) | (s > hi)).sum()
    print(f'  {col:<25}: {n_out:>3} outliers ({n_out/len(s)*100:.1f}%)  '
          f'fence=[{lo:.1f}, {hi:.1f}]')

In [ ]:
# EDA insight 5: Revenue trend theo quarter
print('--- Avg Revenue by Quarter (bn VND) ---')
rev_q = df_fund.groupby('quarter')['revenue_bn_vnd'].mean().round(0)
print(rev_q.to_string())

In [ ]:
# EDA insight 6: Top/bottom tickers by composite score
print('--- Top 5 Tickers by Avg ROE ---')
top_roe = df_fund.groupby('ticker')['roe'].mean().sort_values(ascending=False).head(5)
print((top_roe * 100).round(1).to_string())
print()
print('--- Top 5 Tickers by Avg Net Margin ---')
top_margin = df_fund.groupby('ticker')['net_margin'].mean().sort_values(ascending=False).head(5)
print((top_margin * 100).round(1).to_string())

---
## PHẦN 2: EDA – Dataset Technical

15,120 rows: 30 tickers × 504 ngày giao dịch. OHLCV + 20 chỉ báo kỹ thuật.

In [ ]:
print('=== TECHNICAL DATASET ===')
print(f'Shape: {df_tech.shape}')
print(f'Tickers: {df_tech["ticker"].nunique()}')
print(f'Date range: {df_tech["date"].min()} → {df_tech["date"].max()}')
print(f'Rows per ticker: {len(df_tech) // df_tech["ticker"].nunique()}')
print()
df_tech.head()

In [ ]:
# Missing values – đặc biệt là warmup period MA200
print('--- Missing Values ---')
missing_tech = df_tech.isnull().sum()
missing_tech_pct = (missing_tech / len(df_tech) * 100).round(1)
miss_df = pd.DataFrame({'count': missing_tech, 'pct': missing_tech_pct})
miss_df = miss_df[miss_df['count'] > 0].sort_values('count', ascending=False)
if len(miss_df):
    print(miss_df.to_string())
else:
    print('  Không có missing values.')
print(f'\n  Lý do: Warmup period cho MA200 = 200 rows/ticker × 30 tickers = 6,000 rows null')

In [ ]:
# RSI distribution
print('--- RSI Distribution ---')
rsi = df_tech['rsi_14'].dropna()
oversold = (rsi < 30).sum()
overbought = (rsi > 70).sum()
neutral = ((rsi >= 30) & (rsi <= 70)).sum()
print(f'  Oversold  (RSI < 30):   {oversold:>5} ({oversold/len(rsi)*100:.1f}%)')
print(f'  Neutral   (30-70):      {neutral:>5} ({neutral/len(rsi)*100:.1f}%)')
print(f'  Overbought (RSI > 70):  {overbought:>5} ({overbought/len(rsi)*100:.1f}%)')
print(f'  Mean RSI: {rsi.mean():.2f} | Std: {rsi.std():.2f}')

In [ ]:
# MACD signal distribution
print('--- MACD Distribution ---')
macd_hist = df_tech['macd_histogram'].dropna()
bullish = (macd_hist > 0).sum()
bearish = (macd_hist < 0).sum()
print(f'  Bullish (MACD > 0):  {bullish:>5} ({bullish/len(macd_hist)*100:.1f}%)')
print(f'  Bearish (MACD < 0):  {bearish:>5} ({bearish/len(macd_hist)*100:.1f}%)')

In [ ]:
# Volume spikes
print('--- Volume Spikes (> 2x avg) ---')
vol_ratio = df_tech['volume_ratio'].dropna()
spikes = (vol_ratio > 2.0).sum()
print(f'  Volume spikes: {spikes} ({spikes/len(vol_ratio)*100:.1f}% of days)')
print(f'  Mean volume ratio: {vol_ratio.mean():.2f}')
print(f'  Max volume ratio:  {vol_ratio.max():.2f}')

In [ ]:
# Daily returns
print('--- Daily Returns ---')
daily_ret = df_tech['daily_return'].dropna()
ann_factor = 252  # trading days/year
print(f'  Mean daily return: {daily_ret.mean()*100:.4f}%')
print(f'  Mean annualized:   {daily_ret.mean()*ann_factor*100:.2f}%')
print(f'  Std annualized:    {daily_ret.std()*np.sqrt(ann_factor)*100:.2f}%')
print(f'  Skewness:          {daily_ret.skew():.3f}')
print(f'  Kurtosis:          {daily_ret.kurtosis():.3f}')
print(f'  Negative days:     {(daily_ret < 0).sum()} ({(daily_ret < 0).mean()*100:.1f}%)')

In [ ]:
# Volatility by ticker (top 5 most/least volatile)
print('--- Volatility by Ticker (annualized %) ---')
vol_by_ticker = df_tech.groupby('ticker')['volatility_20d'].mean().sort_values(ascending=False)
print('Top 5 most volatile:')
print((vol_by_ticker.head() * 100).round(1).to_string())
print()
print('Top 5 least volatile:')
print((vol_by_ticker.tail() * 100).round(1).to_string())

In [ ]:
# Golden/Death crosses
print('--- Golden & Death Cross Events ---')
if 'golden_cross' in df_tech.columns and 'death_cross' in df_tech.columns:
    gc = df_tech['golden_cross'].sum()
    dc = df_tech['death_cross'].sum()
    print(f'  Golden Cross events: {gc}')
    print(f'  Death Cross events:  {dc}')
else:
    print('  Columns golden_cross / death_cross not found.')

In [ ]:
# Bollinger Band position distribution
print('--- Bollinger Band Position ---')
bb_pos = df_tech['bb_position'].dropna()
below_lower = (bb_pos < 0).sum()
above_upper = (bb_pos > 1).sum()
inside = ((bb_pos >= 0) & (bb_pos <= 1)).sum()
print(f'  Below lower band (bb_pos < 0):  {below_lower:>5} ({below_lower/len(bb_pos)*100:.1f}%)')
print(f'  Inside bands (0-1):             {inside:>5} ({inside/len(bb_pos)*100:.1f}%)')
print(f'  Above upper band (bb_pos > 1):  {above_upper:>5} ({above_upper/len(bb_pos)*100:.1f}%)')

---
## PHẦN 3: EDA – Dataset Screening

30 cổ phiếu với fundamental + technical scores kết hợp.

In [ ]:
print('=== SCREENING DATASET ===')
print(f'Shape: {df_screen.shape}')
print()
df_screen.head()

In [ ]:
# Top/Bottom tickers by composite score
print('--- Top 10 Tickers by Composite Score ---')
print('  (composite_score = 0.6 × fundamental + 0.4 × technical)')
cols = ['ticker', 'sector', 'composite_score', 'fundamental_score', 'technical_score']
avail = [c for c in cols if c in df_screen.columns]
top10 = df_screen[avail].sort_values('composite_score', ascending=False).head(10)
print(top10.to_string(index=False))

In [ ]:
# Score distribution by sector
print('--- Score by Sector ---')
if 'sector' in df_screen.columns and 'composite_score' in df_screen.columns:
    score_sector = df_screen.groupby('sector')['composite_score'].agg(['mean', 'std', 'count']).round(2)
    print(score_sector.sort_values('mean', ascending=False).to_string())

In [ ]:
# Screening filter example: PE<15 AND ROE>15%
print('--- Screen: PE < 15 AND ROE > 15% ---')
if 'pe_ratio' in df_screen.columns and 'roe' in df_screen.columns:
    filtered = df_screen[(df_screen['pe_ratio'] < 15) & (df_screen['roe'] > 0.15)]
    show_cols = [c for c in ['ticker', 'sector', 'pe_ratio', 'roe', 'composite_score']
                 if c in filtered.columns]
    print(f'  {len(filtered)} tickers passed filter:')
    print(filtered[show_cols].sort_values('composite_score', ascending=False).to_string(index=False))

In [ ]:
# Returns distribution
print('--- Return Distribution ---')
ret_cols = [c for c in df_screen.columns if c.startswith('return_')]
if ret_cols:
    for col in ret_cols:
        s = df_screen[col].dropna()
        print(f'  {col:<15}: mean={s.mean()*100:.1f}%  min={s.min()*100:.1f}%  max={s.max()*100:.1f}%')

In [ ]:
# Trend distribution
print('--- Trend Distribution ---')
if 'trend' in df_screen.columns:
    print(df_screen['trend'].value_counts().to_string())

# RSI distribution in screening snapshot
if 'rsi_14' in df_screen.columns:
    rsi_s = df_screen['rsi_14'].dropna()
    print(f'\nRSI (screening snapshot): mean={rsi_s.mean():.1f}  '
          f'oversold={( rsi_s < 30).sum()}  overbought={(rsi_s > 70).sum()}')

---
## PHẦN 4: EDA – Dataset Orchestrator

140 queries có nhãn để huấn luyện RL orchestrator.

In [ ]:
print('=== ORCHESTRATOR DATASET ===')
print(f'Shape: {df_orch.shape}')
print()
df_orch.head(10)

In [ ]:
# Label distribution
print('--- Agent Label Distribution ---')
print(df_orch['correct_agent'].value_counts().to_string())
print()
# Balanced check
counts = df_orch['correct_agent'].value_counts()
imbalance_ratio = counts.max() / counts.min()
print(f'  Imbalance ratio (max/min): {imbalance_ratio:.2f}x')
if imbalance_ratio > 2:
    print('  WARNING: Significant class imbalance – consider oversampling minority class')
else:
    print('  Class balance OK.')

In [ ]:
# Difficulty distribution
print('--- Difficulty Distribution ---')
print(df_orch['difficulty'].value_counts().to_string())
print()
print('--- Agent × Difficulty Cross-tab ---')
crosstab = pd.crosstab(df_orch['correct_agent'], df_orch['difficulty'])
print(crosstab.to_string())

In [ ]:
# Query length analysis
print('--- Query Length Analysis ---')
df_orch['query_len_chars'] = df_orch['query'].str.len()
df_orch['query_len_words'] = df_orch['query'].str.split().str.len()

ql = df_orch.groupby('correct_agent')[['query_len_chars', 'query_len_words']].agg(['mean', 'max', 'min']).round(1)
print(ql.to_string())
print()
print(f'  Overall avg chars: {df_orch["query_len_chars"].mean():.0f}')
print(f'  Overall avg words: {df_orch["query_len_words"].mean():.0f}')

In [ ]:
# Attachment metadata
print('--- Attachment Info ---')
if 'has_attachment' in df_orch.columns:
    print(df_orch['has_attachment'].value_counts().to_string())
    att_rate = df_orch['has_attachment'].mean() * 100
    print(f'  Attachment rate: {att_rate:.1f}%')

if 'attachment_type' in df_orch.columns:
    print()
    print('Attachment types:')
    print(df_orch['attachment_type'].value_counts(dropna=False).to_string())

In [ ]:
# Keyword frequency analysis
print('--- Top Keywords by Agent ---')
import re
from collections import Counter

def top_words(text_series, top_n=10):
    all_words = []
    for text in text_series:
        words = re.findall(r'[\w/]+', str(text).lower())
        all_words.extend(words)
    counter = Counter(all_words)
    # Lọc stopwords đơn giản
    stopwords = {'của', 'là', 'có', 'trong', 'và', 'cho', 'các', 'không',
                 'cổ', 'phiếu', 'mã', 'cổ phiếu', 'thế', 'nào', 'đang',
                 'với', 'theo', 'qua', 'từ', 'hay', 'như', 'ra', 'tôi'}
    return [(w, c) for w, c in counter.most_common(top_n * 3)
            if w not in stopwords and len(w) > 1][:top_n]

for agent in df_orch['correct_agent'].unique():
    queries = df_orch[df_orch['correct_agent'] == agent]['query']
    kws = top_words(queries)
    print(f'\n  {agent.upper()}:')
    for word, count in kws:
        print(f'    {word:<20}: {count}')

---
## PHẦN 5: Tổng Kết EDA

In [ ]:
print('=' * 65)
print('TỔNG KẾT EDA – KEY INSIGHTS')
print('=' * 65)

insights = [
    ('FUNDAMENTAL', [
        'EPS phân phối lệch phải mạnh (skew~+2.4) → cần log-transform',
        'gross_margin ↔ net_margin: r≈0.85 → multicollinearity, dùng 1 trong 2',
        'PE không thể so sánh cross-sector (Banking PE 5-18, Staples 15-35)',
        'Banking D/E = 5-15 là BÌNH THƯỜNG (mô hình đòn bẩy)',
        'Missing: pb_ratio ~5%, dividend_yield ~3.3% → forward-fill',
    ]),
    ('TECHNICAL', [
        'MA200 null cho 200 ngày đầu/ticker (~39.5% rows) → drop warmup hoặc NaN-aware',
        'RSI cân bằng: 10.8% oversold, 11.3% overbought → thị trường không bias',
        'Volume spikes 7.2% → quan trọng cho detection breakout',
        'Volatility VN ~40.8% annualized (điển hình emerging market)',
        'Daily return mean ~7.6% annualized → tăng trưởng dương',
    ]),
    ('SCREENING', [
        'composite_score = 0.6×fundamental + 0.4×technical',
        'PE<15 AND ROE>15% → chủ yếu Banking (leverage business)',
        'Top performers: MSN, SAB, PLX',
    ]),
    ('ORCHESTRATOR', [
        '140 queries: fundamental(50), screening(50), technical(40) → hơi lệch',
        'Difficulty: hard(54) > medium(55) > easy(31) → dataset bias hard',
        '12.1% queries có attachment metadata',
        'Hard queries thường là mixed-intent → cần multi-agent (Phase 3)',
    ]),
]

for section, items in insights:
    print(f'\n[{section}]')
    for item in items:
        print(f'  • {item}')

print()
print('ACTION ITEMS cho Phase 2:')
actions = [
    'Log-transform EPS trước khi dùng làm feature',
    'Normalize PE theo sector group',
    'Drop 200 warmup rows/ticker trong technical dataset',
    'Oversample technical agent queries (40 → 50) để balance',
    'Label mixed-intent queries với multi-label cho Phase 3',
]
for a in actions:
    print(f'  → {a}')